# Fine-Tuning Qwen3 on Ascend NPUs with Kubeflow Trainer v2 and MindSpeed-LLM

This notebook shows how to run a **Kubeflow Trainer v2** `TrainJob` for Qwen3 fine-tuning on Huawei Ascend NPUs using **MindSpeed-LLM**.

The flow is intentionally close to `qwen3_finetune_verify.ipynb`, but moves the work into a reusable Trainer v2 `TrainingRuntime`:

1. Use the pre-built MindSpeed-LLM NPU runtime image.
2. Create a `TrainingRuntime` that converts Hugging Face weights, preprocesses Alpaca-format data, and launches MindSpeed-LLM SFT.
3. Submit a `TrainJob` that mounts the shared model PVC and requests Ascend resources.
4. Monitor the Trainer v2 job and logs.

The example defaults are smoke-test settings for Qwen3-0.6B: `TRAIN_ITERS=1`, `SEQ_LENGTH=128`, `TP=1`, `PP=1`, and one Ascend 910B4 device. Increase these values for production runs.

## Prerequisites

| Requirement | Example used in this notebook |
|---|---|
| Kubeflow Trainer v2 | `trainer.kubeflow.org/v1alpha1` |
| Namespace | `kubeflow-admin-cpaas-io` |
| Ascend scheduler/runtime | `schedulerName: hami-scheduler`, `runtimeClassName: ascend` |
| Shared model PVC | `team-model-cache-pvc` mounted at `/mnt/models` |
| Base model | `/mnt/models/Qwen3-0.6B` |
| Accelerator resource keys | `huawei.com/Ascend910B4` and `huawei.com/Ascend910B4-memory` |

For larger models, make sure `TP * PP <= NPU count` and that the model architecture arguments in the runtime match the model `config.json`. The provided arguments target Qwen3-0.6B.

## Step 1: Use the Pre-Built Runtime Image

Use the pre-built CANN PyTorch workbench image. It includes the Ascend runtime dependencies and matching versions of `torch`, `torch_npu`, `mindspeed`, and `mindspeed_llm`.

The runtime YAML uses the public image:

```text
alaudadockerhub/alauda-workbench-jupyter-pytorch-cann-py312-ubi9:v0.1.7
```


Important version rule: do not clone `MindSpeed-LLM` HEAD at runtime unless the image was built from the same source revision. The tested path uses installed package modules such as `python -m mindspeed_llm.tasks.checkpoint.convert`, which avoids repo/package drift.

## Step 2: Create the MindSpeed-LLM TrainingRuntime

This runtime contains one `trainer` replicated job. It performs all work in a single NPU pod:

- validates the MindSpeed/PyTorch/NPU environment;
- converts HF weights to Megatron format with `mindspeed_llm.tasks.checkpoint.convert`;
- preprocesses Alpaca-format data with `mindspeed_llm.core.datasets.dataset_preprocess`;
- launches SFT with `torchrun -m mindspeed_llm.tasks.posttrain.launcher`.

The runtime creates a tiny built-in JSONL dataset if `RAW_DATA_FILE` does not already exist. For real training, mount or generate your dataset and override `RAW_DATA_FILE`, `TRAIN_ITERS`, and `SEQ_LENGTH` in the `TrainJob`.

In [ ]:
%%writefile kf-trainingruntime-mindspeed-npu.yaml
apiVersion: trainer.kubeflow.org/v1alpha1
kind: TrainingRuntime
metadata:
  name: mindspeed-llm-qwen3-npu-runtime
  namespace: kubeflow-admin-cpaas-io
  labels:
    trainer.kubeflow.org/framework: torch
spec:
  mlPolicy:
    numNodes: 1
    torch:
      numProcPerNode: auto
  template:
    spec:
      replicatedJobs:
      - name: trainer
        template:
          metadata:
            labels:
              trainer.kubeflow.org/trainjob-ancestor-step: trainer
          spec:
            backoffLimit: 0
            template:
              spec:
                schedulerName: hami-scheduler
                runtimeClassName: ascend
                securityContext:
                  runAsNonRoot: true
                  runAsUser: 1001
                  runAsGroup: 0
                  fsGroup: 1000
                volumes:
                - name: workspace
                  emptyDir: {}
                - name: dshm
                  emptyDir:
                    medium: Memory
                    sizeLimit: 4Gi
                containers:
                - name: node
                  image: alaudadockerhub/alauda-workbench-jupyter-pytorch-cann-py312-ubi9:v0.1.7
                  command: ["bash", "-lc"]
                  args:
                  - |
                    set -o pipefail

                    # The Ascend env scripts may reference unset shell variables or probe
                    # optional libraries. Source them before enabling set -e.
                    set +e
                    for f in /usr/local/Ascend/cann/set_env.sh /usr/local/Ascend/ascend-toolkit/set_env.sh /usr/local/Ascend/nnal/atb/set_env.sh; do
                      [ -f "$f" ] && source "$f"
                    done
                    set -e

                    export CUDA_DEVICE_MAX_CONNECTIONS=1
                    export PYTORCH_NPU_ALLOC_CONF=expandable_segments:True
                    export ASCEND_PROCESS_LOG_PATH=/mnt/workspace/ascendlog
                    mkdir -p "$ASCEND_PROCESS_LOG_PATH"

                    WORK_DIR=${WORK_DIR:-/mnt/workspace/qwen3-0.6b-mindspeed}
                    HF_MODEL_DIR=${HF_MODEL_DIR:-/mnt/models/Qwen3-0.6B}
                    RAW_DATA_FILE=${RAW_DATA_FILE:-${WORK_DIR}/data/alpaca_sample.jsonl}
                    PROCESSED_DATA_PREFIX=${PROCESSED_DATA_PREFIX:-${WORK_DIR}/data/alpaca}
                    MCORE_WEIGHTS_DIR=${MCORE_WEIGHTS_DIR:-${WORK_DIR}/model_weights/qwen3_mcore_tp${TP:-1}_pp${PP:-1}}
                    OUTPUT_DIR=${OUTPUT_DIR:-${WORK_DIR}/output/qwen3_0_6b_finetuned}

                    TP=${TP:-1}
                    PP=${PP:-1}
                    SEQ_LENGTH=${SEQ_LENGTH:-128}
                    TRAIN_ITERS=${TRAIN_ITERS:-1}
                    MBS=${MBS:-1}
                    LR=${LR:-1.25e-6}
                    MIN_LR=${MIN_LR:-1.25e-7}

                    mkdir -p "$(dirname "$RAW_DATA_FILE")" "$MCORE_WEIGHTS_DIR" "$OUTPUT_DIR"
                    if [ ! -s "$RAW_DATA_FILE" ]; then
                      cat >"$RAW_DATA_FILE" <<'JSONL'
{"instruction":"Who are you?","input":"","output":"I am XiaoLing, an AI assistant from Alauda AI Platform.","system":""}
{"instruction":"What is Alauda AI Platform?","input":"","output":"Alauda AI Platform helps teams build, train, and serve AI workloads on Kubernetes.","system":""}
JSONL
                    fi

                    python - <<'PYCHECK'
import importlib.metadata as md
import importlib.util
import torch
import torch_npu
for mod in ["torch", "torch_npu", "mindspeed", "mindspeed_llm"]:
    assert importlib.util.find_spec(mod), f"missing {mod}"
print("torch:", torch.__version__)
print("torch_npu:", torch_npu.__version__)
print("mindspeed:", md.version("mindspeed"))
print("mindspeed_llm:", md.version("mindspeed-llm"))
print("npu_count:", torch.npu.device_count())
assert torch.npu.is_available(), "NPU is not available"
PYCHECK

                    python -m mindspeed_llm.tasks.checkpoint.convert \
                      --load-model-type hf \
                      --save-model-type mg \
                      --target-tensor-parallel-size "$TP" \
                      --target-pipeline-parallel-size "$PP" \
                      --load-dir "$HF_MODEL_DIR" \
                      --save-dir "$MCORE_WEIGHTS_DIR" \
                      --model-type-hf qwen3

                    python -m mindspeed_llm.core.datasets.dataset_preprocess \
                      --input "$RAW_DATA_FILE" \
                      --tokenizer-name-or-path "$HF_MODEL_DIR" \
                      --output-prefix "$PROCESSED_DATA_PREFIX" \
                      --handler-name AlpacaStyleInstructionHandler \
                      --tokenizer-type PretrainedFromHF \
                      --workers 1 \
                      --log-interval 1 \
                      --enable-thinking none \
                      --prompt-type qwen3

                    NPROC=$(python -c 'import torch, torch_npu; print(torch.npu.device_count())')
                    DP=$(( NPROC / (TP * PP) ))
                    GBS=$(( DP * MBS ))
                    [ "$GBS" -ge 1 ] || { echo "Invalid parallelism: NPROC=$NPROC TP=$TP PP=$PP MBS=$MBS"; exit 1; }

                    torchrun \
                      --nproc_per_node "$NPROC" \
                      --nnodes 1 \
                      --node_rank 0 \
                      --master_addr localhost \
                      --master_port 6000 \
                      -m mindspeed_llm.tasks.posttrain.launcher \
                      --use-mcore-models \
                      --spec mindspeed_llm.tasks.models.spec.qwen3_spec layer_spec \
                      --kv-channels 128 \
                      --qk-layernorm \
                      --tensor-model-parallel-size "$TP" \
                      --pipeline-model-parallel-size "$PP" \
                      --sequence-parallel \
                      --use-distributed-optimizer \
                      --use-flash-attn \
                      --num-layers 28 \
                      --hidden-size 1024 \
                      --num-attention-heads 16 \
                      --ffn-hidden-size 3072 \
                      --max-position-embeddings 40960 \
                      --seq-length "$SEQ_LENGTH" \
                      --make-vocab-size-divisible-by 1 \
                      --padded-vocab-size 151936 \
                      --rotary-base 1000000 \
                      --use-rotary-position-embeddings \
                      --micro-batch-size "$MBS" \
                      --global-batch-size "$GBS" \
                      --disable-bias-linear \
                      --swiglu \
                      --train-iters "$TRAIN_ITERS" \
                      --tokenizer-type PretrainedFromHF \
                      --tokenizer-name-or-path "$HF_MODEL_DIR" \
                      --normalization RMSNorm \
                      --position-embedding-type rope \
                      --norm-epsilon 1e-6 \
                      --hidden-dropout 0 \
                      --attention-dropout 0 \
                      --no-gradient-accumulation-fusion \
                      --attention-softmax-in-fp32 \
                      --exit-on-missing-checkpoint \
                      --no-masked-softmax-fusion \
                      --group-query-attention \
                      --num-query-groups 8 \
                      --min-lr "$MIN_LR" \
                      --lr "$LR" \
                      --weight-decay 1e-1 \
                      --clip-grad 1.0 \
                      --adam-beta1 0.9 \
                      --adam-beta2 0.95 \
                      --initial-loss-scale 4096 \
                      --no-load-optim \
                      --no-load-rng \
                      --seed 42 \
                      --bf16 \
                      --data-path "$PROCESSED_DATA_PREFIX" \
                      --split 100,0,0 \
                      --log-interval 1 \
                      --save-interval "$TRAIN_ITERS" \
                      --eval-interval "$TRAIN_ITERS" \
                      --eval-iters 0 \
                      --finetune \
                      --stage sft \
                      --is-instruction-dataset \
                      --prompt-type qwen3 \
                      --no-pad-to-seq-lengths \
                      --distributed-backend nccl \
                      --load "$MCORE_WEIGHTS_DIR" \
                      --save "$OUTPUT_DIR" \
                      --transformer-impl local \
                      --no-save-optim \
                      --no-save-rng
                  env:
                  - name: WORK_DIR
                    value: /mnt/workspace/qwen3-0.6b-mindspeed
                  - name: HF_MODEL_DIR
                    value: /mnt/models/Qwen3-0.6B
                  - name: TP
                    value: "1"
                  - name: PP
                    value: "1"
                  - name: SEQ_LENGTH
                    value: "128"
                  - name: TRAIN_ITERS
                    value: "1"
                  - name: MBS
                    value: "1"
                  securityContext:
                    allowPrivilegeEscalation: true
                    capabilities:
                      add: ["IPC_LOCK", "SYS_PTRACE"]
                    runAsNonRoot: true
                    runAsUser: 1001
                    runAsGroup: 0
                    seccompProfile:
                      type: RuntimeDefault
                  volumeMounts:
                  - name: workspace
                    mountPath: /mnt/workspace
                  - name: dshm
                    mountPath: /dev/shm


In [ ]:
kubectl apply -f kf-trainingruntime-mindspeed-npu.yaml
kubectl get trainingruntime mindspeed-llm-qwen3-npu-runtime -n kubeflow-admin-cpaas-io

## Step 3: Submit a TrainJob

The `TrainJob` mounts the shared PVC at `/mnt/models` and requests one Ascend 910B4 device. The PVC should already contain the Hugging Face Qwen3 model directory used by `HF_MODEL_DIR`.

If your cluster does not expose `huawei.com/Ascend910B4-memory`, remove that resource or replace it with the memory key used by your Ascend device plugin.

In [ ]:
%%writefile kf-trainjob-mindspeed-npu.yaml
apiVersion: trainer.kubeflow.org/v1alpha1
kind: TrainJob
metadata:
  generateName: trainjob-mindspeed-qwen3-
  namespace: kubeflow-admin-cpaas-io
  # If Kueue is enabled, uncomment and set your LocalQueue name.
  # labels:
  #   kueue.x-k8s.io/queue-name: local-queue
spec:
  runtimeRef:
    apiGroup: trainer.kubeflow.org
    kind: TrainingRuntime
    name: mindspeed-llm-qwen3-npu-runtime
  podTemplateOverrides:
  - targetJobs:
    - name: trainer
    spec:
      volumes:
      - name: models-cache
        persistentVolumeClaim:
          claimName: team-model-cache-pvc
      containers:
      - name: node
        volumeMounts:
        - name: models-cache
          mountPath: /mnt/models
  trainer:
    numNodes: 1
    env:
    - name: HF_MODEL_DIR
      value: /mnt/models/Qwen3-0.6B
    - name: TRAIN_ITERS
      value: "1"
    - name: SEQ_LENGTH
      value: "128"
    - name: TP
      value: "1"
    - name: PP
      value: "1"
    resourcesPerNode:
      requests:
        cpu: "4"
        memory: "8Gi"
        huawei.com/Ascend910B4: "1"
        huawei.com/Ascend910B4-memory: "32G"
      limits:
        cpu: "8"
        memory: "32Gi"
        huawei.com/Ascend910B4: "1"
        huawei.com/Ascend910B4-memory: "32G"


In [ ]:
# Use create instead of apply because the TrainJob uses generateName.
kubectl create -f kf-trainjob-mindspeed-npu.yaml

## Step 4: Monitor the Job

Trainer v2 creates a JobSet and one trainer pod for this single-node example.

In [ ]:
kubectl get trainjobs -n kubeflow-admin-cpaas-io
kubectl get jobsets,jobs,pods -n kubeflow-admin-cpaas-io | grep trainjob-mindspeed-qwen3 || true

In [ ]:
# Replace <trainjob-name> with the generated TrainJob name.
kubectl describe trainjob <trainjob-name> -n kubeflow-admin-cpaas-io
kubectl logs -f -n kubeflow-admin-cpaas-io <trainer-pod-name>

## Step 5: Production Adjustments

For a real run, change these values before submitting the `TrainJob`:

| Setting | Where | Guidance |
|---|---|---|
| `TRAIN_ITERS` | `spec.trainer.env` | Increase from `1` to the required training length. |
| `SEQ_LENGTH` | `spec.trainer.env` | Use `4096` or your target context length if memory allows. |
| `TP` / `PP` | runtime env or `TrainJob` env | Match model size and available NPU count. |
| model architecture args | `TrainingRuntime` command | Must match `config.json`; this notebook is for Qwen3-0.6B. |
| dataset | `RAW_DATA_FILE` or PVC content | Use Alpaca JSONL with `instruction`, `input`, `output`, and optional `system`. |
| resources | `resourcesPerNode` | Use the exact Ascend resource keys and memory slices exposed by your cluster. |

For multi-node training, set `spec.trainer.numNodes > 1` only after validating HCCN/device IPs, link state, cross-node reachability, and the HCCL environment for your cluster.

## Step 6: Cleanup

Delete generated TrainJobs when they are no longer needed. Delete the runtime only if no other experiments reference it.

In [ ]:
kubectl delete trainjob <trainjob-name> -n kubeflow-admin-cpaas-io
kubectl delete trainingruntime mindspeed-llm-qwen3-npu-runtime -n kubeflow-admin-cpaas-io

## Validation Notes from NPU Dev

The NPU dev cluster validated the Trainer v2 smoke path with the same PyTorch CANN workbench image family: the job scheduled through `hami-scheduler`, used `runtimeClassName: ascend`, imported `torch`, `torch_npu`, `mindspeed`, and `mindspeed_llm`, and saw one allocated Ascend 910B4 device.

The notebook intentionally uses installed package module entrypoints instead of cloning `MindSpeed-LLM` HEAD at runtime. In NPU dev, cloning HEAD produced a mismatch with installed `mindspeed 0.12.1`. If HAMi reports `CardInsufficientMemory`, wait for other NPU workloads to finish or reduce the requested `huawei.com/Ascend910B4-memory` value according to your cluster policy. If the cluster cannot reach Docker Hub directly, mirror or preload `alaudadockerhub/alauda-workbench-jupyter-pytorch-cann-py312-ubi9:v0.1.7` into a registry reachable from NPU nodes.